In [ ]:
import os

os.getcwd()
# os.path.exists("results/processing_results.json")
json_path = r"/home/wabashcs/abt/video_analysis_pipeline_research-main/results/processing_results.json"  


'/home/wabashcs/abt/video_analysis_pipeline_research-main/notebooks'

In [12]:
import json
import os
import pandas as pd

json_path = r"/home/wabashcs/abt/video_analysis_pipeline_research-main/results/processing_results.json"  

with open(json_path, "r") as f:
    data = json.load(f)

videos = data["results"]

print(f"Videos in results: {len(videos)}")

Videos in results: 1704


In [13]:
records = []

for v in videos:
    extraction = v.get("extraction")

    has_extraction = extraction is not None
    has_error = False
    error_message = None
    error_code = None

    if isinstance(extraction, dict) and "error" in extraction:
        has_error = True
        error_message = extraction.get("error")

        for code in ["429", "504", "503", "500", "403", "401"]:
            if error_message and code in error_message:
                error_code = code
                break

    records.append({
        "video_name": v.get("video_name"),
        "has_extraction": has_extraction,
        "has_error": has_error,
        "error_code": error_code,
    })

df = pd.DataFrame(records)
df.head()


,video_name,has_extraction,has_error,error_code
0,-44_igsZtgU.mp4,False,False,None
1,-1yXdOufzKE.mp4,True,False,None
2,-DFuMqROFCQ.mp4,True,False,None
3,-5X2kzIDroA.mp4,True,False,None
4,-HWd3Nqu8_0.mp4,True,False,None


In [14]:
df["has_extraction"].value_counts()


has_extraction
True     1455
False     249
Name: count, dtype: int64

In [15]:
df["error_code"].value_counts()


error_code
429    108
Name: count, dtype: int64

In [16]:
df[df["error_code"].notna()][["video_name", "error_code"]]


,video_name,error_code
797,TL4LEfRr1TI.mp4,429
798,Ti6saRaKJYM.mp4,429
801,TosjG0E191U.mp4,429
802,TsQXQGaasUg.mp4,429
803,U-VzZQGWOqA.mp4,429
...,...,...
1697,zrIe6n-ZK_E.mp4,429
1699,zvq5rMEbNbw.mp4,429
1701,zl71eFcqBnQ.mp4,429
1702,zwY6acYYO3o.mp4,429


In [17]:
pd.DataFrame([{
    "metadata_total_videos": data["metadata"]["total_videos"],
    "results_count": len(df),
    "videos_with_extraction": df["has_extraction"].sum(),
    "videos_with_errors": df["has_error"].sum(),
}])


,metadata_total_videos,results_count,videos_with_extraction,videos_with_errors
0,1704,1704,1455,167


In [ ]:
# # Filter videos that have an error code
# error_videos = df[df["error_code"].notna()].copy()

# print(f"Videos with errors: {len(error_videos)}")

# # Export to CSV
# output_path = "videos_with_extraction_errors.csv"
# error_videos.to_csv(output_path, index=False)

# output_path


Videos with errors: 108


'videos_with_extraction_errors.csv'

In [20]:
records = []

for v in videos:
    extraction = v.get("extraction")

    error_message = None
    error_code = None

    if isinstance(extraction, dict) and "error" in extraction:
        error_message = extraction["error"]
        for code in ["429", "504", "503", "500", "403"]:
            if code in error_message:
                error_code = code
                break

    if error_code:
        records.append({
            "video_name": v.get("video_name"),
            "video_path": v.get("video_path"),
            "error_code": error_code,
            "error_message": error_message,
            "processing_time_s": v.get("pipeline_stats", {}).get("processing_time_s"),
            "final_frame_count": v.get("pipeline_stats", {}).get("final_frame_count"),
        })

error_df = pd.DataFrame(records)
error_df.to_csv("videos_with_extraction_errors_detailed.csv", index=False)
error_df

,video_name,video_path,error_code,error_message,processing_time_s,final_frame_count
0,TL4LEfRr1TI.mp4,data/hussain_videos/TL4LEfRr1TI.mp4,429,429 Resource has been exhausted (e.g. check qu...,391.733897,110
1,Ti6saRaKJYM.mp4,data/hussain_videos/Ti6saRaKJYM.mp4,429,429 Resource has been exhausted (e.g. check qu...,202.154919,58
2,TosjG0E191U.mp4,data/hussain_videos/TosjG0E191U.mp4,429,429 Resource has been exhausted (e.g. check qu...,21.907001,19
3,TsQXQGaasUg.mp4,data/hussain_videos/TsQXQGaasUg.mp4,429,429 Resource has been exhausted (e.g. check qu...,16.274576,19
4,U-VzZQGWOqA.mp4,data/hussain_videos/U-VzZQGWOqA.mp4,429,429 Resource has been exhausted (e.g. check qu...,11.787360,7
...,...,...,...,...,...,...
103,zrIe6n-ZK_E.mp4,data/hussain_videos/zrIe6n-ZK_E.mp4,429,429 Resource has been exhausted (e.g. check qu...,38.144127,16
104,zvq5rMEbNbw.mp4,data/hussain_videos/zvq5rMEbNbw.mp4,429,429 Resource has been exhausted (e.g. check qu...,10.859097,15
105,zl71eFcqBnQ.mp4,data/hussain_videos/zl71eFcqBnQ.mp4,429,429 Resource has been exhausted (e.g. check qu...,50.662539,24
106,zwY6acYYO3o.mp4,data/hussain_videos/zwY6acYYO3o.mp4,429,429 Resource has been exhausted (e.g. check qu...,17.858696,26
